End-to-End Example for Embedding Creation Using Agent Server Types

This example demonstrates how to:
1. Set up an Azure OpenAI platform client
2. Create embeddings using direct Model selection
3. Create embeddings using ModelSelector (for dynamic model selection)
4. Compare the outputs and understand the embedding response format

To run this example:
1. Setup proper Azure OpenAI credentials in your environment (AZURE_API_KEY, AZURE_ENDPOINT_URL, AZURE_DEPLOYMENT_NAME_EMBEDDINGS, AZURE_API_VERSION)

In [1]:
import os

from utils import setup_notebook

if not setup_notebook(required_keys=[
    "AZURE_API_KEY",
    "AZURE_ENDPOINT_URL",
    "AZURE_DEPLOYMENT_NAME_EMBEDDINGS",
    "AZURE_API_VERSION",
]):
    raise ValueError("Failed to setup notebook, please check your .env file")

In [2]:
import pprint
from typing import Any

import numpy as np

from agent_platform.core.model_selector import (
    DefaultModelSelector,
    ModelSelectionRequest,
)
from agent_platform.core.platforms.azure.client import AzureOpenAIClient
from agent_platform.core.platforms.azure.configs import AzureOpenAIModelMap
from agent_platform.core.platforms.azure.parameters import AzureOpenAIPlatformParameters

pp = pprint.PrettyPrinter(indent=4)

In [3]:
async def create_azure_openai_client() -> AzureOpenAIClient:
    """Create and initialize an Azure OpenAI client.

    Returns:
        An initialized BedrockClient
    """
    # Create the client with credentials
    parameters = AzureOpenAIPlatformParameters(
        azure_api_key=os.getenv("AZURE_API_KEY"),
        azure_endpoint_url=os.getenv("AZURE_ENDPOINT_URL"),
        azure_deployment_name=os.getenv("AZURE_DEPLOYMENT_NAME"),
        azure_deployment_name_embeddings=os.getenv("AZURE_DEPLOYMENT_NAME_EMBEDDINGS"),
        azure_api_version=os.getenv("AZURE_API_VERSION"),
    )
    client = AzureOpenAIClient(
        parameters=parameters,
    )

    print("✅ Azure OpenAI client created successfully")
    return client

In [ ]:
async def run_direct_model_embeddings(
    client: AzureOpenAIClient,
    texts: list[str],
) -> dict[str, Any]:
    """Create embeddings using a direct Model reference.

    Args:
        client: The AzureOpenAIClient instance
        texts: List of text strings to embed

    Returns:
        The embedding response
    """
    print("\n=== Creating Embeddings with Direct Model Reference ===")

    # Select a specific embedding model from our Models catalog
    model = "text-embedding-3-large"
    full_name = AzureOpenAIModelMap
    print(f"Using model: {model} ({full_name})")

    # Create embeddings
    result = await client.create_embeddings(
        texts=texts,
        model=model,
    )

    return result

async def run_model_selector_embeddings(
    client: AzureOpenAIClient,
    texts: list[str],
) -> dict[str, Any]:
    """Create embeddings using a ModelSelector for dynamic model selection.

    Args:
        client: The AzureOpenAIClient instance
        texts: List of text strings to embed

    Returns:
        The embedding response
    """
    print("\n=== Creating Embeddings with ModelSelector ===")

    # Create a model selector with a preference list
    # In a real application, this might include complex selection logic
    model_selector = DefaultModelSelector()

    # The actual model will be selected by the ModelSelector at runtime
    print("Using ModelSelector (will select appropriate model dynamically)")

    # Create embeddings using the model selector
    result = await client.create_embeddings(
        texts=texts,
        model=model_selector.select_model(
            client,
            ModelSelectionRequest(
                model_type="embedding",
                quality_tier="balanced",
            ),
        ),
    )

    # In the actual implementation, the client calls model_selector.select_model()
    # internally to resolve the model_selector to a concrete Model

    return result

In [5]:
def analyze_embedding_results(
    direct_model_result: dict[str, Any],
    selector_result: dict[str, Any],
) -> None:
    """Analyze and compare embedding results from both approaches.

    Args:
        direct_model_result: Result from direct model embeddings
        selector_result: Result from model selector embeddings
    """
    print("\n=== Analyzing Embedding Results ===")

    # Analyze direct model result
    print("\nDirect Model Result:")
    print(f"Model used: {direct_model_result['model']}")
    print(f"Number of embeddings: {len(direct_model_result['embeddings'])}")
    print(f"Embedding dimension: {len(direct_model_result['embeddings'][0])}")
    print(f"Total tokens used: {direct_model_result['usage']['total_tokens']}")

    # Analyze selector result
    print("\nModelSelector Result:")
    print(f"Model used: {selector_result['model']}")
    print(f"Number of embeddings: {len(selector_result['embeddings'])}")
    print(f"Embedding dimension: {len(selector_result['embeddings'][0])}")
    print(f"Total tokens used: {selector_result['usage']['total_tokens']}")

    # Check if results are similar (they should be if the same model was selected)
    models_match = direct_model_result["model"] == selector_result["model"]
    dimensions_match = len(direct_model_result["embeddings"][0]) == len(
        selector_result["embeddings"][0],
    )
    # Check if the embeddings are similar
    # Calculate the cosine similarity between the embeddings
    similarity = np.dot(
        direct_model_result["embeddings"][0], selector_result["embeddings"][0],
    ) / (
        np.linalg.norm(direct_model_result["embeddings"][0])
        * np.linalg.norm(selector_result["embeddings"][0])
    )
    print(f"Cosine similarity between embeddings: {similarity}")

    print("\nComparison:")
    print(f"Same model used: {'✅' if models_match else '❌'}")
    print(f"Same embedding dimensions: {'✅' if dimensions_match else '❌'}")
    print(f"Cosine similarity between embeddings: {similarity}")

In [6]:
async def run_e2e_embeddings():
    """Main function to run the example."""
    print("Starting Embeddings E2E Example...")

    # Sample texts to embed
    sample_texts = [
        "Artificial intelligence is transforming how we work and live.",
        "Machine learning algorithms find patterns in large datasets.",
        "Natural language processing helps computers understand human language.",
    ]

    # Create Bedrock client
    client = await create_azure_openai_client()

    # Create embeddings using both approaches
    direct_result = await run_direct_model_embeddings(client, sample_texts)
    selector_result = await run_model_selector_embeddings(client, sample_texts)

    # Print a sample of the embedding vectors (just first 5 dimensions)
    print("\n=== Sample of Embedding Vectors (first 5 dimensions) ===")
    print("\nDirect Model - First text embedding:")
    pp.pprint(direct_result["embeddings"][0][:5])

    print("\nModelSelector - First text embedding:")
    pp.pprint(selector_result["embeddings"][0][:5])

    # Analyze results
    analyze_embedding_results(direct_result, selector_result)

    print("\n✅ E2E Embeddings Example Complete")

In [ ]:
await run_e2e_embeddings()